# Base Mistral vs NyayaGPT — side-by-side A/B comparison

**Goal:** isolate the effect of fine-tuning by comparing **vanilla Mistral-7B-Instruct-v0.3 (Q4_K_M)** against **NyayaGPT (Q4_K_M)** on the same llama.cpp runtime, same quantization, and same legal questions.

Since Blackwell + CUDA 12.8 breaks every cuBLAS gemm path (FP16, BF16, FP32 all fail per `99_hf_base_inference_test.ipynb`), both models run through llama.cpp's custom CUDA kernels. Using identical quantization (Q4_K_M → Q4_K_M) eliminates quantization as a confounder: every quality delta is attributable to fine-tuning on the 1,521 legal Q&A pairs.

**Prerequisites:**
1. NyayaGPT Q4_K_M GGUF at `adapters/nyayagpt-q4km.gguf` (from Day 4).
2. Base Mistral Q4_K_M GGUF at `/mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf`.
   - If missing, run: `python scripts/build_base_mistral_gguf.py` (~8-10 min, reuses HF cache — no new download).

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

from nyaya_pipeline import config

BASE_GGUF = Path('/mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf')
FT_GGUF   = config.ADAPTER_DIR / 'nyayagpt-q4km.gguf'

assert FT_GGUF.exists(),   f'NyayaGPT GGUF missing at {FT_GGUF}. Run Day 4 first.'
if not BASE_GGUF.exists():
    print(f'⚠ Base Mistral GGUF not found at {BASE_GGUF}')
    print('  Run this in a terminal (takes ~8-10 min):')
    print(f'    cd {PROJECT_ROOT}')
    print(f'    {sys.executable} scripts/build_base_mistral_gguf.py')
else:
    print(f'✓ Base Mistral GGUF: {BASE_GGUF} ({BASE_GGUF.stat().st_size / 1e9:.2f} GB)')
    print(f'✓ NyayaGPT GGUF:     {FT_GGUF} ({FT_GGUF.stat().st_size / 1e9:.2f} GB)')

✓ Base Mistral GGUF: /mnt/f/NyayaGPT-scratch/mistral-base-q4km.gguf (4.37 GB)
✓ NyayaGPT GGUF:     /home/ubuntu/Fine-tuning/NyayaGPT/adapters/nyayagpt-q4km.gguf (4.37 GB)


In [2]:
# ── Load both GGUF models once (reuse across cells) ─────────────────────────
# llama-cpp-python uses its own CUDA kernels — avoids the broken cuBLAS path.
import time
from llama_cpp import Llama

print('Loading base Mistral Q4_K_M … (silent ~20s while weights transfer to GPU)')
base_llm = Llama(
    model_path=str(BASE_GGUF),
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)

print('Loading NyayaGPT Q4_K_M … (silent ~20s while weights transfer to GPU)')
ft_llm = Llama(
    model_path=str(FT_GGUF),
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)

print('✓ Both models loaded.')

Loading base Mistral Q4_K_M … (silent ~20s while weights transfer to GPU)


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Loading NyayaGPT Q4_K_M … (silent ~20s while weights transfer to GPU)


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✓ Both models loaded.


In [3]:
# ── A/B generation helper ───────────────────────────────────────────────────
def compare(question: str, max_new_tokens: int = 256):
    """Run the same question through both models, return (base_resp, ft_resp, latencies)."""
    # Mistral v0.3 chat template with NyayaGPT's domain-specific system prompt.
    # Using the SAME prompt for both isolates fine-tuning as the only variable.
    prompt = f'[INST] {config.SYSTEM_PROMPT}\n\n{question}[/INST]'

    t0 = time.perf_counter()
    base_out = base_llm(prompt, max_tokens=max_new_tokens, temperature=0.0)
    base_ms  = (time.perf_counter() - t0) * 1000
    base_txt = base_out['choices'][0]['text'].strip()
    base_tok = base_out['usage']['completion_tokens']

    t0 = time.perf_counter()
    ft_out = ft_llm(prompt, max_tokens=max_new_tokens, temperature=0.0)
    ft_ms  = (time.perf_counter() - t0) * 1000
    ft_txt = ft_out['choices'][0]['text'].strip()
    ft_tok = ft_out['usage']['completion_tokens']

    return {
        'question':  question,
        'base_text': base_txt, 'base_ms': base_ms, 'base_tokens': base_tok,
        'ft_text':   ft_txt,   'ft_ms':   ft_ms,   'ft_tokens':   ft_tok,
    }

def show(result):
    print(f"Q: {result['question']}")
    print('─' * 78)
    print(f"🔵 Base Mistral   ({result['base_ms']:.0f} ms, {result['base_tokens']} tok)")
    print(f"   {result['base_text']}")
    print()
    print(f"🟢 NyayaGPT       ({result['ft_ms']:.0f} ms, {result['ft_tokens']} tok)")
    print(f"   {result['ft_text']}")
    print('═' * 78)

print('✓ Helpers ready. Use `compare("...")` then `show(result)`.')

✓ Helpers ready. Use `compare("...")` then `show(result)`.


## Single-question demo

In [5]:
result = compare('What are the ingredients of the offence of criminal breach of trust under Indian law?')
show(result)

Q: What are the ingredients of the offence of criminal breach of trust under Indian law?
──────────────────────────────────────────────────────────────────────────────
🔵 Base Mistral   (1356 ms, 256 tok)
   Under Indian law, the offence of Criminal Breach of Trust (CBT) is defined in Section 405 of the Indian Penal Code (IPC). The ingredients of the offence of CBT are as follows:

1. Fiduciary relationship: There must be a relationship of trust and confidence between the person committing the offence (trustee) and the person to whom the trust is owed (creditor or principal). This relationship is known as a fiduciary relationship.

2. Property: The property must be in the possession of the trustee. The property can be movable or immovable, tangible or intangible.

3. Dishonest misappropriation: The trustee must dishonestly misappropriate or convert the property to his own use or to the use of any person other than the creditor.

4. Breach of trust: The misappropriation or conversion of 

## Batch comparison — 5 legal questions

In [6]:
QUESTIONS = [
    'What is the maximum punishment under Section 302 IPC?',
    'Explain Article 21 of the Indian Constitution and its scope.',
    'What are the essential elements of a valid contract under the Indian Contract Act?',
    'What is the difference between cognizable and non-cognizable offences under CrPC?',
    'What remedies are available under the Consumer Protection Act 2019 for defective goods?',
]

batch_results = []
for q in QUESTIONS:
    r = compare(q, max_new_tokens=200)
    batch_results.append(r)
    show(r)
    print()

Q: What is the maximum punishment under Section 302 IPC?
──────────────────────────────────────────────────────────────────────────────
🔵 Base Mistral   (895 ms, 174 tok)
   Section 302 of the Indian Penal Code (IPC) pertains to the offense of murder. The maximum punishment for murder under Section 302 IPC is death sentence, which implies capital punishment, or imprisonment for life, which means imprisonment for the remainder of the convict's natural life. However, it's important to note that the death sentence is not mandatory and is awarded at the discretion of the court.

In the case of Bachan Singh v. State of Punjab (1980), the Supreme Court of India held that the death penalty should be imposed only in the rarest of rare cases to prevent miscarriage of justice and to preserve the sanctity of human life.

For a more detailed understanding, it is recommended to consult with a licensed advocate or legal professional.

🟢 NyayaGPT       (457 ms, 104 tok)
   The maximum punishment for 

In [7]:
# ── Latency summary ─────────────────────────────────────────────────────────
import statistics

base_latencies = [r['base_ms'] / r['base_tokens'] for r in batch_results if r['base_tokens'] > 0]
ft_latencies   = [r['ft_ms']   / r['ft_tokens']   for r in batch_results if r['ft_tokens']   > 0]

print('Per-token latency (ms/tok), averaged across 5 questions')
print('─' * 60)
print(f'Base Mistral (Q4_K_M):  {statistics.mean(base_latencies):6.1f} ms/tok')
print(f'NyayaGPT     (Q4_K_M):  {statistics.mean(ft_latencies):6.1f} ms/tok')
print()
print('Per-question end-to-end')
print('─' * 78)
print(f'{"Question":<50} {"Base (ms)":>10} {"NyayaGPT (ms)":>14}')
print('─' * 78)
for r in batch_results:
    q_trunc = r['question'][:47] + '…' if len(r['question']) > 48 else r['question']
    print(f'{q_trunc:<50} {r["base_ms"]:>10.0f} {r["ft_ms"]:>14.0f}')

Per-token latency (ms/tok), averaged across 5 questions
────────────────────────────────────────────────────────────
Base Mistral (Q4_K_M):     4.6 ms/tok
NyayaGPT     (Q4_K_M):     4.5 ms/tok

Per-question end-to-end
──────────────────────────────────────────────────────────────────────────────
Question                                            Base (ms)  NyayaGPT (ms)
──────────────────────────────────────────────────────────────────────────────
What is the maximum punishment under Section 30…          895            457
Explain Article 21 of the Indian Constitution a…          892            891
What are the essential elements of a valid cont…          900            913
What is the difference between cognizable and n…          910            917
What remedies are available under the Consumer …          911            756


## (Optional) Log this A/B run to MLflow

Each question's latency pair gets logged as a separate run under the `nyayagpt-ab-test` experiment, so the MLflow UI shows base-vs-finetuned directly.

In [8]:
import mlflow

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment(config.MLFLOW_EXPERIMENT_AB)

for r in batch_results:
    with mlflow.start_run(run_name='ab_base_vs_nyaya_q4km'):
        mlflow.log_params({
            'question':         r['question'][:250],
            'base_engine':      'gguf-q4_k_m-vanilla-mistral',
            'finetuned_engine': 'gguf-q4_k_m-nyaya',
            'quantization':     'Q4_K_M (same for both)',
        })
        mlflow.log_metrics({
            'base_latency_ms':      r['base_ms'],
            'finetuned_latency_ms': r['ft_ms'],
            'latency_delta_ms':     r['ft_ms'] - r['base_ms'],
            'base_tokens':          r['base_tokens'],
            'finetuned_tokens':     r['ft_tokens'],
        })

print(f'✓ Logged {len(batch_results)} A/B events to MLflow → {config.MLFLOW_URI}')
print(f'  Open: http://localhost:5000  (experiment: {config.MLFLOW_EXPERIMENT_AB})')

✓ Logged 5 A/B events to MLflow → /home/ubuntu/Fine-tuning/NyayaGPT/mlruns
  Open: http://localhost:5000  (experiment: nyayagpt-ab-test)


/home/ubuntu/Fine-tuning/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


## What to look for

**Quality:** NyayaGPT should cite specific IPC sections, articles, or case law that the base Mistral misses or generalizes.

**Latency:** both are Q4_K_M on the same llama.cpp runtime — expect comparable ms/tok (±10% from random GPU scheduling noise). Any large delta is an artifact, not a real result.

**For LinkedIn:** screenshot a single compelling question (e.g., one citing an obscure case or section) where NyayaGPT demonstrably wins. Pair that screenshot with the Day 4 benchmark chart.